# 3 · Welche Kunden verlieren wir? Ein Churn-Modell
**Päckli & Co.** · Data Science für Marketing · FHNW CAS Data Driven Marketing

**Churn** heisst Abwanderung. Abgewandert = keine Bestellung in den nächsten 90 Tagen. Das Modell lernt am Stichtag 31. Mai 2025 und
bewertet dann alle Kunden zum 31. August 2025.

**Pflicht:** Schritte 3 bis 5 und die Abgabe (7). 1 und 2 sind vorbereitet. **Optional:** Modellvergleich (6, 6b), E1 bis E4.

**Vergleich:** Wer sagt am besten voraus, wer von September bis November 2025 nicht mehr kauft? Am Schluss des Tages vergleichen wir alle Modelle. Nichts zu gewinnen: Ein schwaches Ergebnis ist völlig ok, oft lernen wir daraus am meisten.

**Leakage:** Merkmale nur aus Daten **vor** dem Stichtag. Sonst glänzt das Modell im Test und versagt im Einsatz.

**Probiert es selbst.** Kommt ihr nicht weiter, fragt Gemini (✦ oben rechts) oder einen anderen KI-Assistenten eurer Wahl.
Aber: Ihr müsst jede Zeile eurer Lösung verstehen und erklären können.

**Jeder Schritt:** Code in die leere Zelle › ▶ › **Plausibel?** prüfen. Die Lösung gibt es nach der Übung.
Braucht ein Schritt das Ergebnis eines früheren, der nicht geklappt hat: **▶ Aufholen** ausführen.

**Colab meldet «Zu viele Sitzungen»?** In den Notebooks der früheren Übungen: Laufzeit › «Verbindung trennen und Laufzeit löschen».

In [ ]:
#@title ▶ Setup: tragt eure Teamnummer ein und führt diese Zelle zuerst aus
TEAM = 0  #@param {type:"integer"}
BASIS_URL = "https://raw.githubusercontent.com/badrustanicki/fhnw-ds-marketing-2026/main/"
import os, sys, urllib.request
try:
    urllib.request.urlretrieve(BASIS_URL + "kurs.py", "kurs.py")   # immer die neuste Version
except OSError:
    if not os.path.exists("kurs.py"): raise
sys.modules.pop("kurs", None)   # neu laden, falls schon geladen
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from kurs import *
start(TEAM, BASIS_URL)

### ▶ BigQuery verbinden
Den **Kurs-Schlüssel** einfügen (zeigt die Kursleitung) und ▶. Er gilt nur am Kurstag.

*Nach dem Kurs:* eigenes Gratis-Projekt in [console.cloud.google.com/bigquery](https://console.cloud.google.com/bigquery) anlegen, dessen Projekt-ID statt des Schlüssels einfügen und im Google-Fenster alle Häkchen setzen.

In [ ]:
#@title ▶ BigQuery verbinden
SCHLUESSEL = ""  #@param {type:"string"}
connect_bigquery(SCHLUESSEL, "fhnw-marketing-2026.paeckli")

### ▶ 1 Vorbereitet: Das Label
Einfach ausführen und lesen. Abgewandert (churn = 1) heisst: in den 90 Tagen nach dem Stichtag 31. Mai 2025 keine Bestellung. Ein hoher Anteil? Viele haben nur einmal gekauft.

In [ ]:
label = bq("""
SELECT customer_id,
       IF(COUNTIF(invoice_date >= '2025-06-01' AND invoice_date < '2025-08-30') > 0, 0, 1) AS churn
FROM `fhnw-marketing-2026.paeckli.orders_clean`
GROUP BY customer_id
HAVING COUNTIF(invoice_date < '2025-06-01') > 0
ORDER BY customer_id
""")
print(f"{len(label):,}".replace(",", "'"), f"Kunden, davon abgewandert: {label['churn'].mean():.0%}")

### ▶ 2 Vorbereitet: Merkmale vor dem Stichtag
Einfach ausführen. Die Funktion compute_features(stichtag) berechnet RFM nur aus Bestellungen bis zum Stichtag: So sieht das Modell keine Zukunft (keine Leakage). In Schritt 5 benutzt ihr sie für den 31. August. Die vier weiteren Merkmale sind für den Modellvergleich (6).

In [ ]:
def compute_features(stichtag):
    bis = pd.Timestamp(stichtag) + pd.Timedelta(days=1)
    tag = lambda t: t.strftime("%Y-%m-%d")
    m = bq(f"""
    SELECT customer_id,
           MAX(invoice_date)       AS letzter_kauf,
           MIN(invoice_date)       AS erster_kauf,
           COUNT(DISTINCT invoice) AS frequenz,
           SUM(quantity * price)   AS umsatz,
           -- Bestellungen in den letzten 90 und 365 Tagen
           COUNT(DISTINCT IF(invoice_date >= '{tag(bis - pd.Timedelta(days=90))}', invoice, NULL))  AS bestellungen_90_tage,
           COUNT(DISTINCT IF(invoice_date >= '{tag(bis - pd.Timedelta(days=365))}', invoice, NULL)) AS bestellungen_365_tage,
           -- Bestellungen in den 90 Tagen nach dem Stichtag, aber ein Jahr früher (Saison)
           COUNT(DISTINCT IF(invoice_date >= '{tag(bis - pd.Timedelta(days=365))}'
                         AND invoice_date <  '{tag(bis - pd.Timedelta(days=275))}', invoice, NULL)) AS bestellungen_vorjahr
    FROM `fhnw-marketing-2026.paeckli.orders_clean`
    WHERE invoice_date < '{tag(bis)}'
    GROUP BY customer_id
    ORDER BY customer_id
    """)
    m["tage_seit_kauf"] = (pd.Timestamp(stichtag) - pd.to_datetime(m["letzter_kauf"]).dt.normalize()).dt.days
    m["tage_seit_erstkauf"] = (pd.Timestamp(stichtag) - pd.to_datetime(m["erster_kauf"]).dt.normalize()).dt.days
    return m[["customer_id", "tage_seit_kauf", "frequenz", "umsatz", "tage_seit_erstkauf",
              "bestellungen_90_tage", "bestellungen_365_tage", "bestellungen_vorjahr"]]

train_df = compute_features("2025-05-31").merge(label, on="customer_id")
train_df.describe().round(1)

In [ ]:
print("Zeilen:", len(train_df), "| kleinste tage_seit_kauf:", train_df["tage_seit_kauf"].min())

### 3 Das Modell
**Ziel:** Eine logistische Regression lernt, wie Merkmale und Churn zusammenhängen.

**Aufgabe:** Trainiere mit scikit-learn eine logistische Regression, die churn aus tage_seit_kauf, frequenz und umsatz in train_df vorhersagt. Teile train_df mit train_test_split(train_df, test_size=0.3, random_state=42) in die DataFrames train und test (alle Spalten behalten). Baue eine Pipeline mit drei Schritten: Logarithmus der Merkmale (FunctionTransformer(np.log1p), in der Pipeline, nicht vorher von Hand), StandardScaler, LogisticRegression. Speichere die Pipeline als modell und für die Testkunden die vorhergesagte Wahrscheinlichkeit in test['risiko']. Zeige die Koeffizienten (modell[-1].coef_).

In [ ]:
# Euer Code hier, dann ▶

In [ ]:
#@title ▶ Aufholen: nur ausführen, falls dieser Schritt nicht geklappt hat
if not {"modell", "test"} <= set(globals()) or "risiko" not in test:   # Modell der Musterlösung
    modell, train, test = reference_model(train_df)

**Plausibel?** Mehr Tage seit dem Kauf → höheres Risiko (positiv)? Mehr Bestellungen → tieferes Risiko (negativ)?

### 4 Ist das Modell gut? Lift
**Ziel:** Findet das Modell die Abwanderer besser als der Zufall?

**Aufgabe:** Sortiere die Testkunden in test nach risiko absteigend und nimm die oberen 20 %. Berechne: die Churn-Rate in den oberen 20 %, die Churn-Rate aller Testkunden, den Lift (Rate oben geteilt durch Rate gesamt) und den Anteil aller Abwanderer, der in den oberen 20 % liegt.

In [ ]:
# Euer Code hier, dann ▶

**Plausibel?** Lift über 1 = besser als Zufall. Wandern fast alle ab, ist auch der bestmögliche Lift klein (hier rund 1.3).

### 5 Alle Kunden bewerten (Stichtag 31. August 2025)
**Ziel:** Das Modell auf alle Kunden anwenden: die Grundlage der Kampagne.

**Aufgabe:** Berechne mit compute_features('2025-08-31') die Merkmale aller Kunden und mit modell.predict_proba die Spalte churn_risiko. Ergebnis: DataFrame churn_scores mit customer_id, tage_seit_kauf, frequenz, umsatz, churn_risiko.

In [ ]:
# Euer Code hier, dann ▶

In [ ]:
#@title ▶ Aufholen: nur ausführen, falls dieser Schritt nicht geklappt hat
if "churn_scores" not in globals():   # Ergebnis der Musterlösung
    churn_scores = catch_up("churn_scores", eigene=False)

**Plausibel?** Hohes Risiko bei vielen Tagen seit dem Kauf? Das Modell lernte im Sommer, die Kampagne läuft vor Weihnachten: Was könnte das ändern?

In [ ]:
sanity_check("churn_scores", churn_scores);

### 6 Modellvergleich: Was schlägt die Musterlösung? (optional)
**Ziel:** Modell, Tiefe und Merkmale wählen, ▶, vergleichen. So oft ihr wollt.

**Modelle:** Logistische Regression (Schritt 3), Entscheidungsbaum (eine Folge von Ja/Nein-Fragen),
Random Forest (100 Bäume, die abstimmen). `max_depth` gilt nur für die Bäume: tiefer heisst genauer, aber auch
eher auswendig gelernt.

In [ ]:
#@title 🎚️ Modellvergleich: Regler einstellen, dann ▶
modell_typ = "Logistische Regression"  #@param ["Logistische Regression", "Entscheidungsbaum", "Random Forest"]
max_depth = 3  #@param {type:"slider", min:1, max:20, step:1}
mit_tage_seit_kauf = True  #@param {type:"boolean"}
mit_frequenz = True  #@param {type:"boolean"}
mit_umsatz = True  #@param {type:"boolean"}
mit_tage_seit_erstkauf = False  #@param {type:"boolean"}
mit_bestellungen_90_tage = False  #@param {type:"boolean"}
mit_bestellungen_365_tage = False  #@param {type:"boolean"}
mit_bestellungen_vorjahr = False  #@param {type:"boolean"}

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.tree import DecisionTreeClassifier

ALLE = ["tage_seit_kauf", "frequenz", "umsatz", "tage_seit_erstkauf",
        "bestellungen_90_tage", "bestellungen_365_tage", "bestellungen_vorjahr"]
auswahl = [m for m in ALLE if globals()["mit_" + m]] or ["tage_seit_kauf"]

def new_model(typ):
    if typ == "Entscheidungsbaum":
        return DecisionTreeClassifier(max_depth=max_depth, random_state=0)
    if typ == "Random Forest":
        return RandomForestClassifier(n_estimators=100, max_depth=max_depth, random_state=0, n_jobs=-1)
    return make_pipeline(FunctionTransformer(np.log1p), StandardScaler(), LogisticRegression())

def top_20(df, risiko):   # Churn-Rate unter den 20 % mit dem höchsten Risiko
    return df.assign(r=risiko).nlargest(int(0.2 * len(df)), "r")["churn"].mean()

tr, te = train_test_split(train_df, test_size=0.3, random_state=42)   # dieselbe Aufteilung wie in Schritt 3
tabelle = {}
for name, typ, spalten in [("Musterlösung (Log. Regression, RFM)", "Logistische Regression", ALLE[:3]),
                           ("Euer Modell", modell_typ, auswahl)]:
    m = new_model(typ).fit(tr[spalten], tr["churn"])
    tabelle[name] = {"obere 20 % Training": top_20(tr, m.predict_proba(tr[spalten])[:, 1]),
                     "obere 20 % Test": top_20(te, m.predict_proba(te[spalten])[:, 1])}
if "merkmale_31aug" not in globals():
    merkmale_31aug = compute_features("2025-08-31")
wettbewerb = merkmale_31aug.assign(churn_risiko=m.predict_proba(merkmale_31aug[auswahl])[:, 1])
modell_info = (f"{modell_typ}" + ("" if modell_typ == "Logistische Regression" else f", max_depth {max_depth}")
               + f". Merkmale: {', '.join(auswahl)}")
print("Euer Modell: " + modell_info)

def compare(modell, spalten, abgeben=False):
    """Für Schritt 6b: trainiert euer eigenes Modell und vergleicht es mit der Musterlösung.
    abgeben=True macht es zu eurer Abgabe in Schritt 7."""
    global wettbewerb, modell_info
    tr, te = train_test_split(train_df, test_size=0.3, random_state=42)
    m = modell.fit(tr[spalten], tr["churn"])
    zeile = {"obere 20 % Training": top_20(tr, m.predict_proba(tr[spalten])[:, 1]),
             "obere 20 % Test": top_20(te, m.predict_proba(te[spalten])[:, 1])}
    if abgeben:
        wettbewerb = merkmale_31aug.assign(churn_risiko=m.predict_proba(merkmale_31aug[spalten])[:, 1])
        modell_info = f"Eigene Idee: {type(modell).__name__}. Merkmale: {', '.join(spalten)}"
        print("✅ Das ist jetzt eure Abgabe. Schritt 7 nochmals ausführen.")
    return (pd.DataFrame({"Musterlösung (Log. Regression, RFM)": tabelle["Musterlösung (Log. Regression, RFM)"],
                          "Eigene Idee": zeile}).T * 100).round(1)

(pd.DataFrame(tabelle).T * 100).round(1)

**Plausibel?** Höher ist besser. Liegt euer Test über der Musterlösung? Training viel höher als Test heisst auswendig gelernt. Unterschiede unter etwa 1.5 Punkten sind Zufall. Der Vergleich am Schluss rechnet nicht mit diesem Test, sondern mit den echten Käufen von September bis November 2025.

### 6b Eigene Idee mit Gemini (optional)
**Ziel:** Ein Modell oder ein Merkmal, das die Regler nicht kennen. Zuerst Schritt 6 ausführen.

**Aufgabe:** Berechne in den DataFrames train_df und merkmale_31aug je eine neue Spalte umsatz_pro_bestellung = umsatz / frequenz. Rufe dann die vorhandene Funktion so auf: compare(GradientBoostingClassifier(random_state=0), spalten), mit den Spalten tage_seit_kauf, frequenz, umsatz, umsatz_pro_bestellung und bestellungen_365_tage. Die Funktion trainiert selbst und zeigt das Ergebnis. Die Variable modell aus Schritt 3 nicht überschreiben.

In [ ]:
# Euer Code hier, dann ▶

**Plausibel?** Besser als die Musterlösung und als euer Regler-Modell? Dann mit `compare(..., abgeben=True)` nochmals aufrufen und Schritt 7 ausführen. Eigene Ideen: andere Modelle, andere Merkmale, fragt Gemini.

### ▶ 7 Abgabe (Pflicht)
Ausführen, auch ohne Modellvergleich: Dann zählt euer Modell aus Schritt 5. Mit Modellvergleich zählt der letzte Lauf der Regler (oder von compare(..., abgeben=True)): vor der Abgabe das beste Modell nochmals einstellen. Eine Person pro Team reicht. Abgeben dürft ihr mehrmals, es zählt die letzte Abgabe. Colab fragt nach Google Drive: euer Konto wählen und alles zulassen.

In [ ]:
abgabe = wettbewerb if "wettbewerb" in globals() else churn_scores
info = modell_info if "wettbewerb" in globals() else "Modell aus Schritt 5"
save(abgabe[["customer_id", "churn_risiko"]].assign(modell=info), "churn_wettbewerb")

## Erweiterung (optional)

### E1 Regel gegen Modell
**Ziel:** Ist die Handregel «mindestens 3 Bestellungen, seit 60 Tagen kein Kauf» fast so gut wie das Modell?

**Aufgabe:** Wende auf die Testkunden in test die Handregel an: frequenz mindestens 3 und tage_seit_kauf mindestens 60. Wie viele Kunden erfüllen die Regel, und wie viele davon sind abgewandert (Rate)? Vergleiche mit gleich vielen Testkunden mit dem höchsten risiko.

In [ ]:
# Euer Code hier, dann ▶

**Plausibel?** Wie viel besser ist das Modell? Eine Regel versteht jeder: Lohnt sich der Aufwand?

### E2 Entscheidungsbaum
**Ziel:** Ein kleiner Baum ist leichter zu erklären. Ist er auch gleich gut?

**Aufgabe:** Trainiere auf train einen DecisionTreeClassifier mit max_depth=3 (random_state=0), der churn aus tage_seit_kauf, frequenz und umsatz vorhersagt, zeichne den Baum mit plot_tree und berechne den Lift der oberen 20 % auf test wie in Schritt 4.

In [ ]:
# Euer Code hier, dann ▶

**Plausibel?** Welche Merkmale nutzt der Baum oben?

### E3 Wie tief soll der Baum sein?
**Ziel:** Ein tieferer Baum lernt die Trainingsdaten besser. Sagt er auch neue Kunden besser voraus?

**Aufgabe:** Trainiere den Entscheidungsbaum aus E2 mit max_depth 2, 3, 5 und 10 (random_state=0). Berechne für jede Tiefe die Trefferquote (accuracy) auf train und auf test und den Lift der oberen 20 % auf test. Nimm zum Vergleich als erste Zeile die logistische Regression modell aus Schritt 3 dazu. Zeige alles in einer Tabelle.

In [ ]:
# Euer Code hier, dann ▶

**Plausibel?** Train steigt, Test sinkt: Der tiefe Baum lernt Zufälle auswendig (Folie «Training und Test»). Schlägt ein Baum die logistische Regression? Achtung: «Immer abgewandert» trifft schon 76 %.

### E4 Lift-Kurve
**Ziel:** Den Lift für jeden Anteil angeschriebener Kunden sehen.

**Aufgabe:** Zeichne für test eine Kurve: x = Anteil der Kunden nach risiko sortiert (0 bis 100 %), y = Anteil der erfassten Abwanderer. Zeichne die Diagonale (Zufall) dazu.

In [ ]:
# Euer Code hier, dann ▶

**Plausibel?** Je weiter über der Diagonale, desto mehr bringt das Modell.